In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import finufft
from IPython.display import display, HTML
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
from mpl_toolkits.mplot3d.axes3d import Axes3D
import plotly
import plotly.graph_objects as go
from scipy.stats import wilcoxon

from sptflux import flux_probing, utils

font_path = "/Users/yan569/Library/Fonts/TimesLTStd-Roman.otf"
mpl.font_manager.fontManager.addfont(font_path)
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times LT Std", "Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "text.usetex": True,
})

# example 2d spatiotemporal flux

In [ ]:
# Domain
N = 256
x = np.linspace(0.0, 1.0, N, endpoint=False)
t = np.linspace(0.0, 1.0, N, endpoint=False)
T, X = np.meshgrid(t, x, indexing="ij")

# Define parameters for two Fourier components
amp_mult = 500.0
A1, fx1, ft1 = 2 * amp_mult, 2, 0  # amplitude, spatial freq, temporal freq
A2, fx2, ft2 = 2 * amp_mult, 1, 2
# A3, fx3, ft3 = 1 * amp_mult, 2, 2
# fx=2, ft=0
# fx=1, ft=2

def make_flux_moving_gaussian(
        v, x0=0.10, sigma=0.07,
        A=3000, bg=40,
):

    def flux(X, T):
        u = X - x0 - v*T
        return bg + A * np.exp(-(u**2)/(2*sigma**2))

    return flux

# Construct flux field Φ(x, t)
# def fluxfunc(x, t):
#     z = (
#         A1 * np.cos(2 * np.pi * (fx1 * x + ft1 * t))
#         + A2 * np.cos(2 * np.pi * (fx2 * x + ft2 * t))
#     ) + A1 + A2  # add DC offset
#     return z
v_motion = 0.7  # chosen so (fx,ft)=(4,-2.8) lies on ft=-v fx
fluxfunc = make_flux_moving_gaussian(v=v_motion, x0=0.20, sigma=0.05, A=300000, bg=40)
# flux is T x X order (so imshow will have time on y-axis and space on x-axis)
flux = fluxfunc(X, T)

# ----------------------
# PLOT 1: Spatiotemporal flux field
# ----------------------
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 1, 1, projection="3d")

surf = ax1.plot_surface(
    X, T, flux,
    cmap=cm.inferno,
    linewidth=0, antialiased=False, alpha=1.0
)
ax1.set_xlabel(r"$x$")
ax1.set_ylabel(r"$t$")
ax1.set_zlabel(r"$\varphi(x, t)$")
ax1.set_title(r"Spatiotemporal Flux Field $\varphi(x, t)$")
ax1.set_box_aspect((1, 1, 0.5))
fig.colorbar(surf, ax=ax1, shrink=0.6, pad=0.1, label="Flux (photons/pixel²)")

plt.tight_layout()
plt.savefig("output/plots/sptvis.png", dpi=300)
plt.show()

In [ ]:
plt.imshow(flux, origin="lower")

# overall power spectrum of original flux

In [ ]:
# ----------------------
# PLOT 2: Power Spectrum (|FFT|²)
# ----------------------

F = np.fft.fftshift(np.fft.fft2(flux), axes=(0, 1))
power = np.log1p(np.abs(F))
power = power[9*N//20:11*N//20, 9*N//20:11*N//20]  # Crop to central region

# Frequency coordinates
fx = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))
ft = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))
fx = fx[9*N//20:11*N//20]
ft = ft[9*N//20:11*N//20]
FX, FT = np.meshgrid(fx, ft, indexing="ij")

# Normalize for plotting
# power /= power.max()

# Threshold
threshold = 0.05

fig = plt.figure(figsize=(12, 5))
ax2 = fig.add_subplot(1, 1, 1, projection="3d")
# set a shallower viewing angle
ax2.view_init(elev=8, azim=45)
ax2.plot_surface(FX, FT, power, cmap="magma", linewidth=1, alpha=1.0)
ax2.plot_surface(FX, FT, np.ones_like(power)*threshold, color="cyan", alpha=0.3)

ax2.set_xlabel("Spatial frequency fx")
ax2.set_ylabel("Temporal frequency ft")
ax2.set_zlabel("Power")
ax2.set_title("Flux Power Spectrum |(fx, ft)|")
ax2.set_box_aspect((1, 1, 0.6))

plt.tight_layout()
plt.savefig("output/plots/powercfar.png", dpi=300)
plt.show()

In [ ]:
F = np.fft.fftshift(np.fft.fft2(flux), axes=(0, 1))

In [ ]:
from matplotlib import colors
fx_freqs = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))
ft_freqs = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))
plt.imshow(np.abs(F), norm=colors.LogNorm(vmin=1, vmax=np.abs(F).max()), extent=(fx_freqs.min(), fx_freqs.max(), ft_freqs.min(), ft_freqs.max()), origin="lower")
# zoom in on the region of interest
plt.xlim(-10, 10)
plt.xlabel("fx")
plt.ylabel("ft")
plt.ylim(-10, 10)
plt.colorbar()

In [ ]:
# convert flux to 3 1d arrays of (x, t, flux) for finufft
gridflat_x = X.flatten()
gridflat_t = T.flatten()
gridflat_flux = flux.flatten()
ft_fornufft = np.linspace(ft_freqs.min(), ft_freqs.max(), 5000)
fx_fornufft = np.linspace(fx_freqs.min(), fx_freqs.max(), 5000)
fTT, fXX = np.meshgrid(ft_fornufft, fx_fornufft, indexing="ij")
F_nu = finufft.nufft2d3(
    gridflat_x * 2 * np.pi, gridflat_t * 2 * np.pi,
    gridflat_flux.astype("complex128"),
    fXX.ravel(), fTT.ravel(), eps=1e-6, isign=-1
).reshape(fTT.shape)
F_nu_abs = np.abs(F_nu)

In [ ]:
plt.imshow(F_nu_abs, norm=colors.LogNorm(vmin=1, vmax=np.abs(F_nu).max()), extent=(fx_fornufft.min(), fx_fornufft.max(), ft_fornufft.min(), ft_fornufft.max()), origin="lower")
plt.xlim(-20, 20)
plt.xlabel("fx")
plt.ylabel("ft")
plt.ylim(-20, 20)
plt.colorbar()

# generate points from flux

In [ ]:
# generate inhomogeneous Poisson points using the flux/intensity on the (x,t) grid
# uses existing x, t, fx1, fx2, ft1, ft2 arrays/vars from the notebook

# thinning parameters
lambda_max = flux.max()
area = 1.0  # domain is unit square in (x,t)
random = np.random.default_rng(seed=42)
n_cand = random.poisson(lambda_max * area)

# sample candidate points uniformly on domain
ux = np.random.rand(n_cand)
ut = np.random.rand(n_cand)

lam_vals = fluxfunc(ux, ut)

# accept with probability lam_vals / lambda_max
accept_prob = lam_vals / (lambda_max + 1e-16)
accept = np.random.rand(n_cand) < accept_prob
points_x = ux[accept]
points_t = ut[accept]

scatter_thin = 100

# plot intensity background and accepted points
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111)
im = ax.imshow(
    flux, origin="lower", extent=(0, 1, 0, 1),
    cmap="gray"
)
ax.scatter(points_x[::scatter_thin], points_t[::scatter_thin], s=40, c="r", edgecolor="k", linewidth=1, alpha=0.8)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$t$")
# ax.set_title("Inhomogeneous Poisson points (intensity = flux)")
fig.colorbar(im, ax=ax, label=r"$\varphi(x,t)$")
plt.tight_layout()
plt.savefig("output/plots/spt-flux-both.png", dpi=300)
plt.savefig("output/plots/spt-flux-both.pdf", dpi=300)
plt.show()

print(f"candidates={n_cand}, accepted={points_x.size}")

# multiple ways of visualizing basis functions (i tried multiple ways)

## basis functions by themselves

In [ ]:
# Plot Fourier basis functions (sin(2π fx x) * cos(2π ft t)) for fx = 0..max(fx1,fx2), ft = 0..max(ft1,ft2)
# and overlay the sampled points (ux, ut).
#
# This cell uses variables already defined in the notebook: x, t, ux, ut, fx1, fx2, ft1, ft2

# frequency ranges (inclusive)
fx_max = int(max(fx1, fx2))
fx_max = 2
ft_max = int(max(ft1, ft2))
ft_max = 2
fx_vals = list(range(fx_max + 1))
ft_vals = list(range(ft_max + 1))

# prepare grid for plotting
X, T = np.meshgrid(x, t, indexing="ij")

nrows = len(fx_vals)
ncols = len(ft_vals)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 1.9), squeeze=False)

for i, fx in enumerate(fx_vals):
    for j, ft in enumerate(ft_vals):
        ax = axes[i][j]
        # complex exponential basis, then take real part
        complex_basis = np.exp(-2j * np.pi * (fx * X + ft * T))
        basis = np.real(complex_basis)

        im = ax.imshow(
            basis.T, origin="lower", extent=(0, 1, 0, 1),
            cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal"
        )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"fx={fx}, ft={ft}", fontsize=9)
        ax.set_xlabel("x")
        ax.set_ylabel("t")

# add a colorbar to the right
cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cax, label="basis amplitude")

# fig.suptitle("Fourier basis functions $\cos(2\pi(f_t t + f_x x))$ with sampled points overlaid", fontsize=12)
plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.savefig("output/plots/spt-flux-bases.png", dpi=300)
plt.show()

## basis functions with points on top of them

In [ ]:
# Plot Fourier basis functions (sin(2π fx x) * cos(2π ft t)) for fx = 0..max(fx1,fx2), ft = 0..max(ft1,ft2)
# and overlay the sampled points (ux, ut).
#
# This cell uses variables already defined in the notebook: x, t, ux, ut, fx1, fx2, ft1, ft2

# frequency ranges (inclusive)
fx_max = int(max(fx1, fx2))
fx_max = 2
ft_max = int(max(ft1, ft2))
ft_max = 2
fx_vals = [1, 2]
ft_vals = [0, 1, 2]

# prepare grid for plotting
X, T = np.meshgrid(x, t, indexing="ij")

nrows = len(ft_vals)
ncols = len(fx_vals)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * 1.3, nrows * 1.3),
    squeeze=False,
    sharex=True,
    sharey=True,
    gridspec_kw={"wspace": 0.0, "hspace": 0.17}
)

def eval_basis(x, t, fx, ft):
    complex_basis = np.exp(-2j * np.pi * (fx * x + ft * t))
    return complex_basis

for i, ft in enumerate(ft_vals):
    for j, fx in enumerate(fx_vals):
        ax = axes[i][j]
        # complex exponential basis, then take real part
        complex_basis = eval_basis(X, T, fx, ft)
        basis = np.real(complex_basis)
        basis = np.ones_like(basis)

        im = ax.imshow(
            basis.T, origin="lower", extent=(0, 1, 0, 1),
            cmap="gray", vmin=-1, vmax=1, aspect="equal"
        )

        # overlay sampled points
        single_thin = scatter_thin * 1
        basis_pts_vals_complex = eval_basis(points_x, points_t, fx, ft)
        basis_pts_vals = np.real(basis_pts_vals_complex)
        print(f"fx={fx}, ft={ft} coefficient: {np.sum(basis_pts_vals_complex)}")
        print(f"\tenergy: {np.abs(np.sum(basis_pts_vals_complex))}")

        # basis_pts_vals = np.interp(basis_pts_vals, [-1, 1], [0.3, 1])
        # mask_bad = basis_pts_vals < 0
        # cols = np.zeros((points_x.size, 3))
        # cols[:, 0] = 1.0  # red channel
        # cols[mask_bad] = np.array([68, 129, 227]) / 255.0
        # basis_pts_vals = np.interp(basis_pts_vals, [-1, 1], [0.3, 1])
        # ax.scatter(
        #     points_x[::single_thin], points_t[::single_thin],
        #     s=12, c="r", edgecolor="k", linewidth=0.4, alpha=basis_pts_vals[::single_thin]
        # )

        # mask_bad = basis_pts_vals < 0
        # cols = np.zeros((points_x.size, 3))
        # cols[:, 0] = 1.0  # red channel
        # cols[mask_bad] = np.array([68, 129, 227]) / 255.0
        # ax.scatter(
        #     points_x[::single_thin], points_t[::single_thin],
        #     s=12, c=cols[::single_thin], edgecolor="k", linewidth=0.4, alpha=0.8
        # )

        cols = utils.vals_to_cmap(basis_pts_vals, cmap="bwr", vmin=-1, vmax=1)
        alphas = np.interp(basis_pts_vals, [-1, 1], [0.6, 1])[::single_thin]
        sca = ax.scatter(
            points_x[::single_thin], points_t[::single_thin],
            s=12, c=cols[::single_thin], edgecolor="k", linewidth=0.0, alpha=0.2
        )

        # sca = ax.scatter(
        #     points_x[::single_thin], points_t[::single_thin],
        #     s=12, c="r", edgecolor="k", linewidth=0.4, alpha=0.2
        # )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(fr"($f_x,f_t)=({fx},{ft})$", fontsize=9, pad=3)
        # ax.set_xlabel(r"$x$")
        # ax.set_ylabel(r"$t$")

# shared axis labels (only once)
fig.supxlabel(r"$x$", fontsize=11, y=0.07)
fig.supylabel(r"$t$", fontsize=11, x=0.07)

# add a colorbar to the right
cax = fig.add_axes([0.92, 0.3, 0.02, 0.4])
cbar = fig.colorbar(
    cm.ScalarMappable(mpl.colors.Normalize(vmin=-1, vmax=1), cmap="bwr"),
    cax=cax, label="event strength"
)
cbar.set_label("event strength", labelpad=-10)
cbar.set_ticks([-1.0, 1.0])
# fig.colorbar(im, cax=cax, label="Basis amplitude")

# fig.suptitle(r"Fourier basis functions $\cos(2\pi(f_t t + f_x x))$ with sampled points overlaid", fontsize=12)
# plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.savefig("output/plots/bases-pts.pdf", dpi=300, bbox_inches="tight")
plt.savefig("output/plots/bases-pts.png", dpi=300, bbox_inches="tight")
plt.show()

## alternatives to visualizing the strength of each basis (these probably look better?)

In [ ]:
# ------------------------------------------------------------
# FIX 1 — Local Accumulated Coefficient Map (Heatmap)
# ------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

def basis_val(fx, ft, x, t):
    return np.exp(-2j * np.pi * (fx * x + ft * t))

# Grid for binning points (same resolution as flux grid)
N = 25
X, T = np.meshgrid(x, t, indexing="ij")

# Digitize points onto the grid
ix = np.minimum((points_x * N).astype(int), N-1)
it = np.minimum((points_t * N).astype(int), N-1)

fig, axes = plt.subplots(len(ft_vals), len(fx_vals),
                         figsize=(1.8*len(fx_vals), 1.8*len(ft_vals)),
                         squeeze=False)

# vmin = 0
# vmax = 0
# for i, ft in enumerate(ft_vals):
#     for j, fx in enumerate(fx_vals):
#         C = np.zeros((N, N), dtype=complex)

#         # Accumulate complex basis contributions into bins
#         vals = basis_val(fx, ft, points_x, points_t)
#         for k in range(points_x.size):
#             C[ix[k], it[k]] += vals[k]
#             if np.max(np.real(C)) > vmax:
#                 vmax = np.max(np.real(C))
#             if np.min(np.real(C)) < vmin:
#                 vmin = np.min(np.real(C))
# absmax = max(abs(vmin), abs(vmax))
# vmin = -absmax
# vmax = absmax

for i, ft in enumerate(ft_vals):
    for j, fx in enumerate(fx_vals):
        C = np.zeros((N, N), dtype=complex)

        # Accumulate complex basis contributions into bins
        vals = basis_val(fx, ft, points_x, points_t)
        for k in range(points_x.size):
            C[ix[k], it[k]] += vals[k]
        absmax = max(np.abs(np.real(C)).max(), np.abs(np.real(C)).min())
        vmin = -absmax
        vmax = absmax

        ax = axes[i][j]
        im = ax.imshow(np.real(C).T, origin='lower',
                       extent=(0,1,0,1), cmap='RdBu_r', vmin=vmin, vmax=vmax)
        ax.set_title(fr"$(f_x,f_t)=({fx},{ft})$")
        ax.set_xticks([]); ax.set_yticks([])

# Colorbar
cax = fig.add_axes([1, 0.4, 0.02, 0.2])
cbar = fig.colorbar(
    cm.ScalarMappable(mpl.colors.Normalize(vmin=-1, vmax=1), cmap="RdBu_r"),
    cax=cax, label="normalized real pixel energy"
)
cbar.set_ticks([-1.0, 1.0])
plt.savefig("output/plots/bases-pts-dens.pdf", dpi=300, bbox_inches="tight")
plt.savefig("output/plots/bases-pts-dens.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------
# FIX 3 — Kernel-Smoothed Similarity Fields
# ------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

def basis_val(fx, ft, x, t):
    return np.exp(-2j * np.pi * (fx * x + ft * t))

N = 25
X, T = np.meshgrid(x, t, indexing="ij")
ix = np.minimum((points_x * N).astype(int), N-1)
it = np.minimum((points_t * N).astype(int), N-1)
cmap = "RdBu_r"
sigma = 1.2  # smoothing (in grid pixels)

fig, axes = plt.subplots(len(ft_vals), len(fx_vals),
                         figsize=(1.8*len(fx_vals), 1.8*len(ft_vals)),
                         squeeze=False)
# fighist, axeshist = plt.subplots(len(ft_vals), len(fx_vals))

# maxenergy = 0
# absmax = 0
# for i, ft in enumerate(ft_vals):
#     for j, fx in enumerate(fx_vals):
#         if ft == 0 and fx == 0:
#             continue
#         vals = basis_val(fx, ft, points_x, points_t)
#         # Scatter contributions onto a grid
#         S = np.zeros((N, N), float)
#         vals = np.real(basis_val(fx, ft, points_x, points_t))
#         energy = np.abs(np.sum(vals))
#         for k in range(points_x.size):
#             S[ix[k], it[k]] += vals[k]
#         print(energy)
#         if energy > maxenergy:
#             maxenergy = energy
#         absmax_new = max(np.abs(S).max(), np.abs(S).min())
#         if absmax_new > absmax:
#             absmax = absmax_new
# vmin = -absmax
# vmax = absmax

for i, ft in enumerate(ft_vals):
    for j, fx in enumerate(fx_vals):
        # Scatter contributions onto a grid
        S = np.zeros((N, N), float)
        vals_complex = basis_val(fx, ft, points_x, points_t)
        vals = np.real(vals_complex)
        vals_imag = np.imag(vals_complex)
        energy = np.abs(np.sum(vals))

        for k in range(points_x.size):
            S[ix[k], it[k]] += vals[k]
        print(f"freq fx,fy={fx, ft}")
        # axhist = axeshist[i, j]
        # axhist.hist(vals.flatten(), bins=30)
        # axhist.set_title(fr"Real event energies for $(f_x,f_t)=({fx},{ft})$")
        print(f"\tmedian real: {np.median(vals.flatten())}")
        print(f"\tpval real: {wilcoxon(vals.flatten())}")
        print(f"\tpval imag: {wilcoxon(vals_imag.flatten())}")

        # Smooth the field
        S_smooth = gaussian_filter(S, sigma=sigma)
        absmax = max(np.abs(S_smooth).max(), np.abs(S_smooth).min())
        vmin = -absmax
        vmax = absmax

        ax = axes[i][j]
        # im = ax.imshow(S_smooth.T * energy / maxenergy, origin='lower', extent=(0,1,0,1),
        #                cmap=cmap, vmin=vmin, vmax=vmax)
        im = ax.imshow(S_smooth.T, origin='lower', extent=(0,1,0,1),
                       cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(fr"$(f_x,f_t)=({fx},{ft})$")
        ax.set_xticks([]); ax.set_yticks([])

cax = fig.add_axes([1, 0.4, 0.02, 0.2])
cbar = fig.colorbar(
    cm.ScalarMappable(mpl.colors.Normalize(vmin=-1, vmax=1), cmap=cmap),
    cax=cax, label="normalized real pixel energy"
)
cbar.set_ticks([-1.0, 1.0])
plt.tight_layout()
plt.savefig("output/plots/bases-pts-gaussdens.pdf", dpi=300, bbox_inches="tight")
plt.savefig("output/plots/bases-pts-gaussdens.png", dpi=300, bbox_inches="tight")
plt.show()


# attempt at visualizing points on top of flux in 3D (look bad)

In [ ]:
# 3D Plotly surface of the flux with event scatter overlay
# Uses variables already defined in the notebook: X, T, flux, points_x, points_t, fluxfunc, go

# compute event heights on the continuous flux function
z_pts = fluxfunc(points_x, points_t) + 100.0  # offset above surface for visibility

fig = go.Figure()

# surface (use the meshgrid arrays so axes are scaled correctly)
fig.add_trace(go.Surface(
    x=X, y=T, z=flux,
    colorscale="greys_r",
    colorbar=dict(title="flux"),
    opacity=1.0,
    name="flux surface",
    showscale=True
))

# scatter events on top of the surface
fig.add_trace(go.Scatter3d(
    x=points_x, y=points_t, z=z_pts,
    mode="markers",
    marker=dict(size=3, color="red", opacity=0.8),
    name="events",
    hovertemplate="x=%{x:.3f}<br>t=%{y:.3f}<br>flux=%{z:.2f}<extra></extra>"
))

fig.update_layout(
    title="Flux Surface with Event Locations",
    width=900, height=700,
    scene=dict(
        xaxis_title="x",
        yaxis_title="t",
        zaxis_title=r"\varphi(x,t)",
        camera=dict(eye=dict(x=1.6, y=1.2, z=0.6))
    )
)

# save static images (requires kaleido/plotly image support)
try:
    fig.write_image("output/plots/flux_surface_events.png", width=900, height=700, scale=300/72)
    fig.write_image("output/plots/flux_surface_events.pdf", width=900, height=700, scale=300/72)
except Exception:
    # ignore if image backend not available in the environment
    pass

fig.show()

# showing actual UWB on 2d data

In [ ]:
F_uwb = finufft.nufft2d3(
    points_x * 2 * np.pi, points_t * 2 * np.pi,
    np.ones(len(points_x), dtype="complex128"),
    fXX.ravel(), fTT.ravel(), eps=1e-6, isign=-1
).reshape(fTT.shape)
F_uwb_abs = np.abs(F_uwb)
# Threshold
threshold = flux_probing.compute_amplitude_bound(
    len(points_x), V=1, alpha=0.01
)
mask = F_uwb_abs > threshold
F_uwb_bd = F_uwb.copy()
F_uwb_bd[~mask] = 0.0

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(
    np.abs(F_uwb_bd),
    norm=colors.LogNorm(vmin=1, vmax=np.abs(F_uwb_abs).max()),
    extent=(fx_fornufft.min(), fx_fornufft.max(), ft_fornufft.min(), ft_fornufft.max()),
    origin="lower",
)
axes[0].set_xlim(-10, 10)
axes[0].set_ylim(-10, 10)
axes[0].set_xlabel("fx")
axes[0].set_ylabel("ft")
axes[0].set_title("UWB CFAR alpha=0.01")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(
    F_nu_abs,
    norm=colors.LogNorm(vmin=1, vmax=np.abs(F_nu).max()),
    extent=(fx_fornufft.min(), fx_fornufft.max(), ft_fornufft.min(), ft_fornufft.max()),
    origin="lower",
)
axes[1].set_xlim(-10, 10)
axes[1].set_ylim(-10, 10)
axes[1].set_xlabel("fx")
axes[1].set_ylabel("ft")
axes[1].set_title("GT magnitude")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
plt.imshow(F_nu_abs, norm=colors.LogNorm(vmin=1, vmax=np.abs(F_nu).max()), extent=(fx_fornufft.min(), fx_fornufft.max(), ft_fornufft.min(), ft_fornufft.max()), origin="lower")
plt.xlim(-10, 10)
plt.xlabel("fx")
plt.ylabel("ft")
plt.ylim(-10, 10)
plt.colorbar()

In [ ]:
# # ----------------------
# # PLOT 2: Power Spectrum (|FFT|²)
# # ----------------------

# fx_slice = flux_probing.center_slices(N, 10, symmetric=True)[0]
# ft_slice = flux_probing.center_slices(N, 10, symmetric=True)[0]
# power = np.abs(F_nu)
# power = power[fx_slice, ft_slice]  # Crop to central region

# # Frequency coordinates
# fx = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))
# ft = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))
# fx = fx[fx_slice]
# ft = ft[ft_slice]
# FX, FT = np.meshgrid(fx, ft, indexing="ij")

# # Normalize for plotting
# # power /= power.max()

# fig = plt.figure(figsize=(12, 5))
# ax2 = fig.add_subplot(1, 1, 1, projection="3d")
# # set a shallower viewing angle
# # ax2.view_init(elev=35, azim=45)
# ax2.plot_surface(FX, FT, np.ones_like(power)*np.log1p(threshold), color="cyan", alpha=0.5)
# ax2.plot_surface(FX, FT, np.log1p(power), cmap="magma", linewidth=1, alpha=1.0)

# ax2.set_xlabel("$f_x$")
# ax2.set_ylabel("$f_t$")
# ax2.set_zlabel("$A_{(f_x,f_t)}$")
# # ax2.set_title("Flux Power Spectrum |Φ̂(fx, ft)|²")
# # ax2.set_box_aspect((1, 1, 0.6))

# # plt.tight_layout()
# plt.savefig("output/plots/powercfar.png", dpi=300)
# plt.show()

## cfar bound in 3d plot (looks bad)

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ----------------------
# PLOT 2: Power Spectrum (|FFT|²)
# ----------------------

nfreqs = 10
fx_slice = flux_probing.center_slices(N, nfreqs, symmetric=True)[0]
ft_slice = flux_probing.center_slices(N, nfreqs, symmetric=True)[0]
power = np.abs(F_nu)
power = power[fx_slice, ft_slice]  # crop to central region
power = power[:, nfreqs//2:]

# Frequency coordinates
fx = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))[fx_slice]
ft = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))[ft_slice][nfreqs//2:]
FX, FT = np.meshgrid(fx, ft, indexing="ij")

Z = np.log1p(power)
Zthr = np.ones_like(Z) * np.log1p(threshold)

# ----------------------
# PLOTLY FIGURE
# ----------------------

fig = go.Figure()

# Main power spectrum surface
fig.add_trace(go.Surface(
    x=FX, y=FT, z=Z,
    colorscale="gray",
    colorbar=dict(title="log(1 + power)"),
    showscale=True,
    name="Power"
))

# Threshold plane
fig.add_trace(go.Surface(
    x=FX, y=FT, z=Zthr,
    opacity=0.6,
    colorscale=[[0, "cyan"], [1, "cyan"]],
    showscale=False,
    name="Threshold"
))

fig.update_layout(
    width=900, height=650,
    scene=dict(
        xaxis_title="$f_x$",
        yaxis_title="$f_t$",
        zaxis_title=r"$A_{f_x, f_t}$",
        camera=dict(eye=dict(x=1.6, y=1.4, z=0.5))
    )
)
# save at 300 DPI (scale relative to 72 DPI baseline)
fig.write_image("output/plots/powercfar_plotly.png", width=900, height=650, scale=300/72)
fig.write_image("output/plots/powercfar_plotly.pdf", width=900, height=650, scale=300/72)
fig.show()


In [ ]:
N=256
F_nu_bd = F_nu.copy()
mask = np.abs(F_nu_bd) < threshold
F_nu_bd[mask] = 0
print(f"freqs above threshold: {np.sum(~mask)}")
out = np.fft.irfftn(np.fft.ifftshift(F_nu_bd), axes=(0, 1), s=(N, N), norm="forward")
flux_recon = np.clip(out.reshape(N, N).real, 0, None)

In [ ]:
fx_full = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))
ft_full = np.fft.fftshift(np.fft.fftfreq(N, d=1/N))

In [ ]:
detect_where = np.where(~mask)
detectedamps = np.abs(F_nu[detect_where])
detectedfreqs = np.array([fx_full[detect_where[0]], ft_full[detect_where[1]]]).T
detectedfreqs = detectedfreqs[detectedfreqs[:,1]>=0]
detectedfreqs, detectedamps

In [ ]:
# detectedfreqs = [
#     [2, 12]
# ]

In [ ]:
import os

outdir = "output/plots/bases_detected"
os.makedirs(outdir, exist_ok=True)

for k, (fx_val, ft_val) in enumerate(detectedfreqs):
    fx_i = int(fx_val)
    ft_i = int(ft_val)

    complex_basis = np.exp(-2j * np.pi * (fx_i * X + ft_i * T))
    basis = np.real(complex_basis)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(
        basis.T, origin="lower", extent=(0, 1, 0, 1),
        cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal"
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("x")
    ax.set_ylabel("t")

    fname = os.path.join(outdir, f"basis_fx{fx_i:+d}_ft{ft_i:+d}.png")
    plt.tight_layout()
    plt.savefig(fname, dpi=300)
    plt.close(fig)

# complex_basis = np.exp(2j * np.pi * (-17 * X + 94 * T))
# basis = np.real(complex_basis)
# fig, ax = plt.subplots(figsize=(4, 4))
# im = ax.imshow(
#     basis.T, origin="lower", extent=(0, 1, 0, 1),
#     cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal"
# )

# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_xlabel("x")
# ax.set_ylabel("t")

In [ ]:
plt.imshow(flux_recon.T, origin="lower", extent=(0, 1, 0, 1), cmap="gray")
plt.colorbar(label=r"$\hat\varphi(x,t)$")
plt.xlabel(r"$x$")
plt.ylabel(r"$t$")
# plt.title("Reconstructed Flux Field from Thresholded NUFFT Coefficients")
plt.tight_layout()
plt.savefig("output/plots/spt-flux-recon.png", dpi=300)
plt.savefig("output/plots/spt-flux-recon.pdf", dpi=300)
plt.show()

In [ ]:
plt.imshow(flux.T, origin="lower", extent=(0, 1, 0, 1), cmap="gray")
plt.colorbar(label=r"$\varphi(x,t)$")
plt.xlabel(r"$x$")
plt.ylabel(r"$t$")
# plt.title("GT Flux Field")
plt.tight_layout()
plt.savefig("output/plots/spt-flux.png", dpi=300)
plt.savefig("output/plots/spt-flux.pdf", dpi=300)
plt.show()

In [ ]:
diff = np.abs(flux - flux_recon)
plt.imshow(diff.T, origin="lower", extent=(0, 1, 0, 1), cmap="magma")
plt.colorbar(label="Absolute Difference")

In [ ]:
import numpy as np

def psnr(img1, img2, max_val=None):
    img1 = np.asarray(img1, dtype=float)
    img2 = np.asarray(img2, dtype=float)

    # If max_val not provided, infer from data
    if max_val is None:
        max_val = max(img1.max(), img2.max())

    mse = np.mean((img1 - img2) ** 2)

    if mse == 0:
        return np.inf

    return 20 * np.log10(max_val) - 10 * np.log10(mse)
psnr_val = psnr(flux, flux_recon, max_val=flux.max())
print(f"PSNR between original and reconstructed flux: {psnr_val:.2f}")

# misc testing idk

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import get_window

# -----------------------------
# 1. Create piecewise sine wave
# -----------------------------
fs = 64            # sampling rate
T = 3                # total duration in seconds
N = fs * T
t = np.linspace(0, T, N, endpoint=False)

# Three frequencies for three time intervals
f1, f2, f3 = 2, 5, 10

signal = np.zeros_like(t)
segment = N // 3

signal[:segment] = np.sin(2*np.pi*f1 * t[:segment])
signal[segment:2*segment] = np.sin(2*np.pi*f2 * t[segment:2*segment])
signal[2*segment:] = np.sin(2*np.pi*f3 * t[2*segment:])

# -----------------------------
# 2. Global Fourier Transform
# -----------------------------
freqs = np.fft.rfftfreq(N, 1/fs)
fft_mag = np.abs(np.fft.rfft(signal))**2

# -----------------------------
# 3. Windowed STFT-like analysis
# -----------------------------
win_len = 8
hop = win_len // 2          # 50% overlap
window = get_window("hann", win_len)

num_frames = (N - win_len) // hop + 1
stft_mag = []

for i in range(num_frames):
    start = i * hop
    frame = signal[start:start + win_len] * window
    F = np.abs(np.fft.rfft(frame))**2
    stft_mag.append(F)

stft_mag = np.array(stft_mag)         # shape: (frames, freq_bins)
times = np.arange(num_frames) * hop / fs
f_windowed = np.fft.rfftfreq(win_len, 1/fs)

# -----------------------------
# 4. Plotting
# -----------------------------
plt.figure(figsize=(12, 9))

# Original signal
plt.subplot(3,1,1)
plt.plot(t, signal)
plt.title("Original Piecewise Sine Wave")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

# Global Fourier Transform
plt.subplot(3,1,2)
plt.plot(freqs, fft_mag)
plt.title("Global Fourier Transform (Periodogram)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")

# Windowed STFT Periodograms
plt.subplot(3,1,3)
plt.imshow(
    10*np.log10(stft_mag.T + 1e-6),
    aspect='auto',
    origin='lower',
    extent=[times[0], times[-1], f_windowed[0], f_windowed[-1]]
)
plt.colorbar(label="Power (dB)")
plt.title("Hann Windowed Periodograms (STFT)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")

plt.tight_layout()
plt.show()


In [ ]:
# Python code to demonstrate why STFT (windowed Fourier) is useful.
# This will create five separate figures (each its own matplotlib figure):
# 1) Time-domain piecewise signal with 3 sections
# 2) Periodogram from a single Fourier transform of the whole signal
# 3) STFT spectrogram (Hann window, 50% overlap)
# 4) Average of the windowed periodograms (Welch-like average) compared with the full-signal periodogram
# 5) Reconstruction from the full-signal FFT after zeroing low-power frequency bins
#
# Notes:
# - Uses numpy and matplotlib only (no seaborn, no explicit color choices).
# - Window length chosen so time-frequency tradeoff is visible.
# - One section intentionally uses a non-differentiable square wave to produce a "messy" Fourier transform.

import numpy as np
import matplotlib.pyplot as plt

# Sampling parameters
fs = 512  # Hz
sec_per_section = 1.0  # seconds for each piece
n_sections = 3
T = sec_per_section * n_sections
t = np.arange(0, T, 1/fs)
N = len(t)

# Build piecewise signal (three 1-second sections)
# Section 1: clean sine at 10 Hz
t1 = t[(t >= 0) & (t < sec_per_section)]
s1 = np.sin(2 * np.pi * 10 * t1)

# Section 2: non-differentiable square wave at 20 Hz (many harmonics -> messy FT)
t2 = t[(t >= sec_per_section) & (t < 2*sec_per_section)]
s2 = np.sign(np.sin(2 * np.pi * 20 * (t2 - sec_per_section)))
# make sure mean is zero
s2 = s2 - np.mean(s2)

# Section 3: two close sinusoids (30 Hz + 40 Hz) to show multiple tones
t3 = t[(t >= 2*sec_per_section) & (t < 3*sec_per_section)]
s3 = 0.7 * np.sin(2 * np.pi * 30 * (t3 - 2*sec_per_section)) + 0.5 * np.sin(2 * np.pi * 40 * (t3 - 2*sec_per_section))

# Concatenate to full signal, add small white noise
signal = np.concatenate([s1, s2, s3])
signal = signal + 0.02 * np.random.randn(len(signal))  # small noise to make it realistic

# Plot 1: time-domain signal with boundaries
plt.figure(figsize=(10, 3.2))
plt.plot(t, signal)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Piecewise signal (3 sections: 10Hz sine | 20Hz square | 30Hz+40Hz sum)')
# Mark section boundaries
for b in [sec_per_section, 2*sec_per_section]:
    plt.axvline(b, linestyle='--')
plt.xlim(0, T)
plt.tight_layout()

# ------------------------
# Full-signal Fourier periodogram
# ------------------------
# use rfft for positive frequencies only
fft_full = np.fft.rfft(signal)
freqs = np.fft.rfftfreq(N, 1/fs)
power_full = (np.abs(fft_full) ** 2) / N  # power spectral density estimate (non-windowed)

# Convert to dB for plotting (use small epsilon to avoid log(0))
eps = 1e-12
power_full_db = 10 * np.log1p(power_full)

plt.figure(figsize=(10, 3.2))
# plt.semilogx(freqs, power_full_db)
plt.plot(freqs, power_full_db)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power (dB)')
plt.title('Full-signal periodogram (single FFT of entire signal)')
plt.xlim(0, 100)  # show 10 Hz to Nyquist on log scale for visibility
plt.grid(True, which='both', linestyle=':', linewidth=0.5)
plt.tight_layout()

# ------------------------
# STFT (Hann window, 50% overlap)
# ------------------------
window_len = 128  # samples (~0.256 s)
hop = window_len // 2  # 50% overlap
window = np.hanning(window_len)

# pad signal to fit an integer number of frames
n_frames = 1 + (N - window_len) // hop if (N - window_len) >= 0 else 0
# if not enough frames, make sure at least one frame
if n_frames <= 0:
    n_frames = 1
    pad = window_len - N
    sig_padded = np.concatenate([signal, np.zeros(pad)])
else:
    total_needed = window_len + (n_frames - 1) * hop
    pad = total_needed - N
    if pad > 0:
        sig_padded = np.concatenate([signal, np.zeros(pad)])
    else:
        sig_padded = signal.copy()

frames = []
for i in range(n_frames):
    start = i * hop
    frame = sig_padded[start:start + window_len] * window
    frames.append(frame)
frames = np.array(frames)  # shape (n_frames, window_len)

# compute rfft for each frame
stft = np.fft.rfft(frames, axis=1)
pow_stft = (np.abs(stft) ** 2) / window_len  # power per-frame (normalized by window length)
# shape: (n_frames, n_freqs). We'll transpose for plotting: (n_freqs, n_frames)
pow_stft_T = pow_stft.T
freqs_stft = np.fft.rfftfreq(window_len, 1/fs)
times_stft = (np.arange(n_frames) * hop + window_len/2) / fs  # center time of each frame (approx.)

# Plot 3: spectrogram (power in dB)
pow_stft_db = 10 * np.log1p(pow_stft_T)

plt.figure(figsize=(10, 3.2))
# imshow requires (rows=freqs, cols=times)
extent = [times_stft[0] - (hop/fs)/2, times_stft[-1] + (hop/fs)/2, freqs_stft[0], freqs_stft[-1]]
plt.imshow(pow_stft_db, aspect='auto', origin='lower', extent=extent)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('STFT spectrogram (Hann window, 50% overlap) — power (dB)')
plt.ylim(0, 100)
plt.colorbar(label='Power (dB)')
plt.tight_layout()

# ------------------------
# Average windowed periodogram (Welch-like) vs Full FFT
# ------------------------
avg_pow = np.mean(pow_stft_T, axis=1)  # average over time frames
avg_pow_db = 10 * np.log1p(avg_pow)

plt.figure(figsize=(10, 3.2))
plt.plot(freqs_stft, avg_pow_db, label='Average windowed periodogram (Hann, 50% overlap)')
# overlay full-signal periodogram but resample or align frequency axes: freq_stft covers 0..fs/2
# Interpolate full power onto stft frequency bins
from numpy import interp
power_full_on_stft = interp(freqs_stft, freqs, power_full)
power_full_on_stft_db = 10 * np.log1p(power_full_on_stft)
plt.plot(freqs_stft, power_full_on_stft_db, linestyle='--', label='Full-signal FFT periodogram')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power (dB)')
plt.title('Average of windowed periodograms (Welch-like) vs Full FFT periodogram')
plt.xlim(0, 500)
plt.legend()
plt.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()

# ------------------------
# Reconstruction from the thresholded full-signal spectrum
# ------------------------
# Keep only the frequency bins whose power exceeds a fraction of the peak power,
# zero the rest, and invert the FFT.
power_thresh = 0.01 * power_full.max()  # -20 dB relative to the strongest bin
power_thresh_db = 10
keep = power_full_db >= power_thresh_db
fft_thresh = np.where(keep, fft_full, 0)
signal_recon = np.fft.irfft(fft_thresh, n=N)
print(f"kept {keep.sum()} of {len(freqs)} bins (power >= {power_thresh_db:.3g})")

plt.figure(figsize=(10, 3.2))
# plt.plot(t, signal, linewidth=0.8, label='Original')
plt.plot(t, signal_recon, label=f'Reconstruction ({keep.sum()} bins kept)')
for b in [sec_per_section, 2*sec_per_section]:
    plt.axvline(b, linestyle='--')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Reconstruction from full-signal FFT with low-power bins zeroed')
plt.xlim(0, T)
# plt.legend()
plt.tight_layout()

# Show all plots
plt.show()

# Print a short summary of the analysis so the user can interpret the figures
print("Summary:")
print("- Top plot: the piecewise time-domain signal with three sections.")
print("- Second plot: periodogram computed from a single FFT of the entire signal. Notice contributions from every section mixed together.")
print("- Third plot: STFT spectrogram using Hann windows and 50% overlap. Localized time-frequency structure is visible (e.g., square-wave harmonics only present in the middle section).")
print("- Fourth plot: average of the windowed periodograms (Welch-like average) compared to the full-signal FFT. The windowed average often reduces spectral leakage and reveals clearer peaks for persistent tones, while the full FFT mixes everything together.")
print("- Fifth plot: signal reconstructed from the full-signal FFT after zeroing every frequency bin below 1% of the peak power. The surviving global sinusoids run across the whole record, so the piecewise structure (and the square wave's edges) is not recovered.")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# COLA demonstration with Hann windows, 50% overlap
N = 256                # window length
hop = N // 2           # 50% overlap
n_windows = 6          # number of overlapping windows to visualize

# Hann window
w = np.hanning(N+1)[:-1]

# Time axis for full plotting region
total_len = N + (n_windows - 1) * hop
t = np.arange(total_len)

# Container to accumulate overlap-add
ola = np.zeros(total_len)

plt.figure(figsize=(10, 4))

# Plot each shifted window
for i in range(n_windows):
    start = i * hop
    end = start + N
    
    # Add to overlap-add buffer
    ola[start:end] += w
    
    # Plot the individual window (same color, low alpha)
    plt.plot(t[start:end], w, c="b", alpha=0.5)

# Plot the OLA sum
plt.plot(t, ola, linewidth=3, c="r", label="SUM of overlapping windows")

plt.title("Hann Window COLA Property (50% Overlap)")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True, linestyle=':')

plt.tight_layout()
plt.show()

print("Min sum:", np.min(ola))
print("Max sum:", np.max(ola))
print("They should be equal (constant).")


In [ ]:
import plotly.graph_objects as go
import numpy as np

# Example data
x = np.linspace(0, 10, 50)
y = np.linspace(0, 10, 50)
X, Y = np.meshgrid(x, y)
Z = np.sin(X) * np.cos(Y)

fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y)])

# LaTeX in axis titles
fig.update_layout(
    scene=dict(
        xaxis_title=r"$X\ \mathrm{[m]}$",
        yaxis_title=r"$Y\ \mathrm{[m]}$",
        zaxis_title=r"$Z = \sin(X)\cos(Y)$"
    )
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

z1 = np.array([
    [8.83,8.89,8.81,8.87,8.9,8.87],
    [8.89,8.94,8.85,8.94,8.96,8.92],
    [8.84,8.9,8.82,8.92,8.93,8.91],
    [8.79,8.85,8.79,8.9,8.94,8.92],
    [8.79,8.88,8.81,8.9,8.95,8.92],
    [8.8,8.82,8.78,8.91,8.94,8.92],
    [8.75,8.78,8.77,8.91,8.95,8.92],
    [8.8,8.8,8.77,8.91,8.95,8.94],
    [8.74,8.81,8.76,8.93,8.98,8.99],
    [8.89,8.99,8.92,9.1,9.13,9.11],
    [8.97,8.97,8.91,9.09,9.11,9.11],
    [9.04,9.08,9.05,9.25,9.28,9.27],
    [9,9.01,9,9.2,9.23,9.2],
    [8.99,8.99,8.98,9.18,9.2,9.19],
    [8.93,8.97,8.97,9.18,9.2,9.18]
])

z2 = z1 + 1
z3 = z1 - 1

fig = go.Figure(data=[
    go.Surface(z=z1),
    go.Surface(z=z2, showscale=False, opacity=0.9),
    go.Surface(z=z3, showscale=False, opacity=0.9)

])

fig.show()

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111)
ax.imshow(np.log1p(power), origin="lower", cmap="magma")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the continuous flux function Φ(x, y, t)
def flux(x, y, t):
    return np.sin(2 * np.pi * x) * np.cos(2 * np.pi * t) + np.cos(2 * np.pi * y) * np.sin(2 * np.pi * t)

# Sampling resolution for cube faces
N = 100
u = np.linspace(0, 1, N)
v = np.linspace(0, 1, N)
U, V = np.meshgrid(u, v)

# Evaluate flux on each of the six cube faces
faces = []

# x = 0 and x = 1
faces.append((np.zeros_like(U), U, V, flux(0, U, V)))
faces.append((np.ones_like(U), U, V, flux(1, U, V)))

# y = 0 and y = 1
faces.append((U, np.zeros_like(U), V, flux(U, 0, V)))
faces.append((U, np.ones_like(U), V, flux(U, 1, V)))

# t = 0 and t = 1
faces.append((U, V, np.zeros_like(U), flux(U, V, 0)))
faces.append((U, V, np.ones_like(U), flux(U, V, 1)))

# Normalize color range
all_flux_vals = np.concatenate([f[3].ravel() for f in faces])
vmin, vmax = all_flux_vals.min(), all_flux_vals.max()

# Plot cube with colored faces
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")

for X, Y, T, F in faces:
    color_vals = (F - vmin) / (vmax - vmin)
    ax.plot_surface(
        X, Y, T,
        facecolors=plt.cm.coolwarm(color_vals),
        rstride=1, cstride=1,
        linewidth=0, antialiased=False, shade=False
    )

# Aesthetics
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("time t")
ax.set_box_aspect((1, 1, 1))
ax.set_title("Spatiotemporal Flux Field Φ(x, y, t) on Unit Cube Surfaces")
ax.view_init(25, 45)
ax.grid(False)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define Fourier components
def phi_x(x, y, t):
    return np.sin(2 * np.pi * x) * np.cos(2 * np.pi * t)

def phi_y(x, y, t):
    return np.cos(2 * np.pi * y) * np.sin(2 * np.pi * t)

# Sampling resolution for cube faces
N = 100
u = np.linspace(0, 1, N)
v = np.linspace(0, 1, N)
U, V = np.meshgrid(u, v)

def make_faces(func):
    """Evaluate a scalar field func(x,y,t) on all six cube faces."""
    faces = []
    # x = 0, 1
    faces.append((np.zeros_like(U), U, V, func(0, U, V)))
    faces.append((np.ones_like(U), U, V, func(1, U, V)))
    # y = 0, 1
    faces.append((U, np.zeros_like(U), V, func(U, 0, V)))
    faces.append((U, np.ones_like(U), V, func(U, 1, V)))
    # t = 0, 1
    faces.append((U, V, np.zeros_like(U), func(U, V, 0)))
    faces.append((U, V, np.ones_like(U), func(U, V, 1)))
    return faces

# Compute faces for both components
faces_x = make_faces(phi_x)
faces_y = make_faces(phi_y)

# Color normalization across both fields for comparability
all_vals = np.concatenate([f[3].ravel() for f in faces_x + faces_y])
vmin, vmax = all_vals.min(), all_vals.max()

# Plot setup
fig = plt.figure(figsize=(12, 6))

for i, (faces, title) in enumerate(
    [(faces_x, r"$\Phi_x(x,t) = \sin(2\pi x)\cos(2\pi t)$"),
     (faces_y, r"$\Phi_y(y,t) = \cos(2\pi y)\sin(2\pi t)$")],
    start=1,
):
    ax = fig.add_subplot(1, 2, i, projection="3d")

    for X, Y, T, F in faces:
        color_vals = (F - vmin) / (vmax - vmin)
        ax.plot_surface(
            X, Y, T,
            facecolors=plt.cm.coolwarm(color_vals),
            rstride=1, cstride=1,
            linewidth=0, antialiased=False, shade=False
        )

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("time t")
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(25, 45)
    ax.grid(False)

plt.tight_layout()
plt.show()


# intensity vs poisson spectra vis

In [ ]:
imgs = utils.read_imgdir("data/phantom-synth/separated_videos/stirfry-fps1000", grayscale=True)

import matplotlib.pyplot as plt
from pathlib import Path

outdir = Path("output/poster-vis")
outdir.mkdir(parents=True, exist_ok=True)

nframes = 41  # frames 0-40 inclusive
# imgs is (T, H, W)
frame_idxs = [0, 20, 40]


def save_frames(vol, prefix, idxs=frame_idxs, outdir=outdir):
    """Save frames at `idxs` as pngs, sharing one vmax across the set."""
    vmax = max(vol[i].max() for i in idxs)
    for i in idxs:
        plt.imsave(outdir / f"{prefix}_frame{i}.png", vol[i], cmap="gray", vmin=0, vmax=vmax)


# save the original frames 0, 20, 40 as pngs
save_frames(imgs, "orig")

def save_tf_power_spectra(vol, prefix, outdir=outdir, nslices=3):
    """vol: (T, H, W) with time on axis 0.

    Do a 3D rfftn with time as the real (rfft) axis: full complex FFT over the
    spatial (H, W) axes, real FFT over time. Passing axes=(1, 2, 0) makes the
    time axis last in the axes list, so rfft is applied to it and axis 0 of the
    output holds the time frequencies. Save the log1p power spectra of the first
    `nslices` spatial slices along the time-frequency axis (spatially fftshifted),
    sharing one vmax across the set so the slices are comparable.
    """
    F = np.fft.rfftn(vol, axes=(1, 2, 0))  # -> (nfreq_t, H, W)
    powers = [np.log1p(np.abs(np.fft.fftshift(F[k])) ** 2) for k in range(nslices)]
    vmax = max(p.max() for p in powers)  # shared scale across the 3 slices
    for k, power in enumerate(powers):
        plt.imsave(outdir / f"{prefix}_tf{k}.png", power, cmap="magma", vmin=0, vmax=vmax)
    return F


# original intensity frames -> time-frequency power spectra
F_intensity = save_tf_power_spectra(imgs[:nframes].astype(np.float64), "intensity")

# Poisson-sample the frames relative to their intensity to simulate what a
# single-photon (SPAD) camera reads, at two thinning levels.
rng = np.random.default_rng(0)
norm = imgs[:nframes] / 255.0  # intensity in [0, 1]

poisson_levels = {
    # more photons: multi-photon counts, generally >1 photon/px but not binary
    "poisson_hi": dict(scale=20.0, clip=False),
    # fewer photons, clipped to 1 -> binary single-photon frames
    "poisson_lo": dict(scale=0.5, clip=True),
}

for prefix, cfg in poisson_levels.items():
    counts = rng.poisson(norm * cfg["scale"]).astype(np.float64)
    if cfg["clip"]:
        counts = np.clip(counts, 0, 1)  # binary: at most 1 photon per pixel
    print(f"{prefix}: mean={counts.mean():.3f} photons/px, max={counts.max():.0f}")
    save_frames(counts, prefix)
    save_tf_power_spectra(counts, prefix)


# barebones velocity plane vis

In [ ]:
"""
Visualize the spectral support of a constant-velocity moving pattern:

    v_x*f_x + v_y*f_y + f_t = 0

This is a plane through the origin in (f_x, f_y, f_t) space, tilted
according to the velocity (v_x, v_y).
"""

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Times New Roman",
    "font.size": 20,
})

# ---------------------------------------------------------------- settings
extent = 1.0            # half-width of the fx,fy grid
n = 25                  # grid resolution
speed = 0.6             # velocity magnitude used for the tilted cases
wireframe = False       # draw grid wireframe on the plane
show_ft0_plane = False  # draw a transparent reference plane at ft = 0
fg = 'white'            # axes / outline / text color (for a dark slide)
plane_color = '#8ab6ff'  # lighter blue for the tilted plane


# ---------------------------------------------------------------- geometry
lin = np.linspace(-extent, extent, n)
fx, fy = np.meshgrid(lin, lin)
qx, qy = np.meshgrid([-extent, extent], [-extent, extent])


def bare_axes(ax, L):
    """Draw plain fx, fy, ft axis lines through the origin, nothing else."""
    ax.set_axis_off()
    ax.plot([-L, L], [0, 0], [0, 0], color=fg, lw=1)
    ax.plot([0, 0], [-L, L], [0, 0], color=fg, lw=1)
    ax.plot([0, 0], [0, 0], [-L, L], color=fg, lw=1)
    ax.text(L * 1.2, 0, 0, r'$f_x$', ha='center', va='center', color=fg)
    ax.text(0, L * 1.1, 0, r'$f_y$', ha='center', va='center', color=fg)
    ax.text(0, 0, L * 1.15, r'$f_t$', ha='center', va='center', color=fg)
    ax.set_xlim(-L, L)
    ax.set_ylim(-L, L)
    ax.set_zlim(-L, L)


def plot_plane(ax, vx, vy, wireframe=False, show_ft0_plane=False):
    # tilted plane: vx*fx + vy*fy + ft = 0  ->  ft = -(vx*fx + vy*fy)
    ft = -(vx * fx + vy * fy)
    # single quad (qx, qy) rather than the n x n grid: the translucent quad
    # seams are invisible on white but show as a lattice on black.
    # shade=False: the default shading darkens the face, which reads as dark
    # rather than light against a dark background
    ax.plot_surface(qx, qy, -(vx * qx + vy * qy), color=plane_color, alpha=0.8,
                    shade=False, linewidth=0, antialiased=True)
    if wireframe:
        ax.plot_wireframe(fx, fy, ft, color=fg, linewidth=0.4, alpha=0.6,
                          rcount=10, ccount=10)
    # outline around the plane: perimeter of the fx,fy square lifted onto it
    e = extent
    cx = np.array([-e, e, e, -e, -e])
    cy = np.array([-e, -e, e, e, -e])
    ax.plot(cx, cy, -(vx * cx + vy * cy), color=fg, lw=1.2)
    if show_ft0_plane:
        # transparent reference plane at ft = 0 (the fx, fy plane)
        ax.plot_surface(fx, fy, np.zeros_like(fx), color='gray', alpha=0.2,
                        linewidth=0, antialiased=True)
        ax.plot(cx, cy, np.zeros_like(cx), color='gray', lw=1.0)
    # dashed line where the tilted plane meets ft = 0: vx*fx + vy*fy = 0,
    # i.e. the line through the origin perpendicular to (vx, vy)
    if vx != 0 or vy != 0:
        dx, dy = -vy, vx
        t = e / max(abs(dx), abs(dy))
        ax.plot([-t * dx, t * dx], [-t * dy, t * dy], [0, 0],
                color=fg, lw=1.2, ls='--')
    bare_axes(ax, extent * 1.3)
    ax.view_init(elev=18, azim=-20)


# ---------------------------------------------------------------- three planes
cases = [
    (0.0, speed, 'vx0'),      # vx = 0
    (speed, 0.0, 'vy0'),      # vy = 0
    (speed, speed, 'vxvy'),   # vx = vy
    (0, 0, '00'),
]
for vx, vy, tag in cases:
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(1, 1, 1, projection='3d')
    plot_plane(ax, vx, vy, wireframe=wireframe, show_ft0_plane=show_ft0_plane)
    fig.tight_layout()
    fig.savefig(f'output/poster-vis/motion_plane_{tag}.svg', dpi=300,
                transparent=True)
    plt.close(fig)


# binary video saving for website/video

In [ ]:
import numpy as np
from spadlib.video_io import write_video
from spadlib.io import read_quanta_auto
from pathlib import Path

# datapath = Path("/Volumes/CrucialX10/uwb3d-files/data/quanta/Feb29_balloonbounce_acq0_2875ppps.zarr")
datapath = Path("/Volumes/CrucialX10/uwb3d-files/data/quanta/Feb27_blender_acq15_4351ppps.zarr")
frames, _ = read_quanta_auto(
    datapath,
    load_data=False
)
# frames = frames[:]
# frames = frames[18500:28100]
frames = frames[65000:75000]

In [ ]:
write_video(
    frames,
    f"/Volumes/CrucialX10/uwb3d-files/data/binary-vids/binary-{datapath.stem}.mp4",
    playback_fps=30, cmap="gray"
)

In [ ]:
# at the higher fps, choose a subset of frames
# subsamp_frames = frames[:]
subsamp_frames = frames[18500:28100]
sum_every = 10
sum_frames = []
for i in range(0, len(subsamp_frames), sum_every):
    chunk = subsamp_frames[i:i + sum_every]
    if np.shape(chunk)[0] < sum_every:
        break
    summed = chunk.sum(axis=0)
    sum_frames.append(summed)
sum_frames = np.array(sum_frames)
write_video(
    sum_frames,
    f"/Volumes/CrucialX10/uwb3d-files/data/accum-vids/accumframes-Feb29_balloonbounce_acq0_2875ppps-{sum_every}.mp4",
    playback_fps=30, cmap="gray", gamma=1/2.2
)

## exposure-ladder / growing-exposure videos with burned-in overlays

Helpers: a streaming writer (so the overlay video never has to live in RAM) and a
blocked accumulator over the lazy zarr array. No gamma: at these exposures a gamma
curve crushes the few grey levels there are and 10k fps ends up looking purely binary.
`UPSCALE` enlarges the frames by nearest neighbour, which leaves the photon data
untouched and only exists so the overlay text gets drawn at a higher resolution.

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm
from spadlib.io import read_quanta_auto
from spadlib.video_io import get_codec_for_format
from pathlib import Path

# ZARR = "/Volumes/CrucialX10/uwb3d-files/data/quanta/Feb29_balloonbounce_acq0_2875ppps.zarr"
ZARR = "/Volumes/CrucialX10/uwb3d-files/data/quanta/Feb27_blender_acq15_4351ppps.zarr"
ZARR = Path(ZARR)
OUTDIR = "/Volumes/CrucialX10/uwb3d-files/data/accum-vids"
OUTDIR = Path(OUTDIR)
NATIVE_FPS = 100_000     # 100k binary frames, i.e. 1.0 s of acquisition
PLAYBACK_FPS = 30
UPSCALE = 2              # nearest-neighbour spatial upscale, purely so the text renders
                         # at a higher resolution; 1 leaves the frames at native size
FONT = cv2.FONT_HERSHEY_DUPLEX


def format_us(t_us, compact=False):
    """Duration as '123 ms + 450 us'; compact drops the '0 ms + ' below one ms."""
    t_us = int(round(t_us))
    if compact and t_us < 1000:
        return f"{t_us} us"
    return f"{t_us // 1000} ms + {t_us % 1000:03d} us"


def format_fps(fps):
    """Frame rate with 3 significant digits, e.g. '33.3k fps', '200 fps'."""
    if fps >= 1000:
        return f"{float(f'{fps / 1000:.3g}'):g}k fps"
    return f"{float(f'{fps:.3g}'):g} fps"


class OverlayWriter:
    """Streaming grayscale writer, small text burned into the bottom-right corner.

    Frames are passed in as mean photons per native frame, so every segment of the
    video shares one brightness scale and only the noise level changes.
    """

    def __init__(self, path, H, W, scale, fps=PLAYBACK_FPS, upscale=UPSCALE):
        self.scale = scale
        self.u = upscale
        self.size = (W * upscale, H * upscale)
        fourcc = cv2.VideoWriter_fourcc(*get_codec_for_format("mp4"))
        self.vw = cv2.VideoWriter(str(path), fourcc, fps, self.size, isColor=True)
        self.n = 0

    def frame(self, rate, lines):
        """rate: (H, W) mean photons per native frame. lines: list of (text, fontscale)."""
        img = (np.clip(rate / self.scale, 0, 1) * 255).astype(np.uint8)
        if self.u != 1:
            img = cv2.resize(img, self.size, interpolation=cv2.INTER_NEAREST)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        self._overlay(img, lines)
        return img

    def write(self, rate, lines, repeat=1):
        img = self.frame(rate, lines)
        for _ in range(repeat):
            self.vw.write(img)
            self.n += 1

    def _overlay(self, img, lines, pad=7, gap=5):
        # every metric scales with the upscale factor, so the overlay keeps the same
        # size relative to the frame but is drawn with more pixels to render it
        u = self.u
        pad, gap, thick = pad * u, gap * u, max(1, round(u))
        H, W = img.shape[:2]
        sizes = [cv2.getTextSize(t, FONT, fs * u, thick)[0] for t, fs in lines]
        block_h = sum(s[1] for s in sizes) + gap * (len(lines) - 1)
        x0 = W - pad - max(s[0] for s in sizes)
        y = H - pad - block_h
        panel = img[y - pad:H, x0 - pad:W]
        panel[:] = (0.3 * panel).astype(np.uint8)  # darken behind the text
        for (text, fs), (tw, th) in zip(lines, sizes):
            y += th
            cv2.putText(img, text, (W - pad - tw, y), FONT, fs * u, (255, 255, 255),
                        thick, cv2.LINE_AA)
            y += gap

    def close(self):
        self.vw.release()
        return self.n


def accumulate(frames, start, count, block=1500):
    """Sum native frames [start, start+count) as float32, reading in blocks."""
    acc = np.zeros(frames.shape[1:], dtype=np.float32)
    i = start
    while i < start + count:
        j = min(i + block, start + count)
        acc += frames[i:j].sum(axis=0, dtype=np.float32)
        i = j
    return acc

One streaming pass over the whole acquisition gives a 200 fps base (sums of 500 native
frames). Both videos reuse it: the 30/200 fps segments are built entirely from it, and
its full sum sets the shared brightness scale. ~40 s on the external SSD.

In [ ]:
BIN = 500  # native frames per base (200 fps) bin

quanta, quanta_md = read_quanta_auto(ZARR, load_data=False)   # lazy zarr, (100000, 512, 512)
T, H, W = quanta.shape
nbins = T // BIN

base200 = np.zeros((nbins, H, W), dtype=np.float32)
for b in tqdm(range(nbins), desc="200 fps base"):
    base200[b] = accumulate(quanta, b * BIN, BIN, block=BIN)

# shared brightness scale: a high quantile of the fully-accumulated photon rate. q0.995
# without gamma keeps the long exposures from blowing out while leaving the short ones
# with more than a couple of usable grey levels
scale = float(np.quantile(base200.sum(0) / T, 0.995))
print(f"{T} frames @ {NATIVE_FPS} fps, brightness scale = {scale:.4f}")

### 1. exposure ladder: 30 -> 200 -> 1k -> 10k -> 50k -> 100k fps

30 and 200 fps play the acquisition all the way through (every native frame is used);
the rest are windows centred on the pop, cut instantly between rates.

100000/30 is not an integer, so the 30 fps segment groups the 200 fps base into 6- or
7-bin chunks (33.3 or 28.6 ms) and normalises by the bin count, which leaves brightness
unaffected. It is labelled nominally as 30 fps.

In [ ]:
# CENTER = 23_300   # native frame at the centre of the action
CENTER = 70_000   # native frame at the centre of the action
N_OUT = 120       # output frames per windowed segment (4 s at 30 fps playback)

ladder_path = f"{OUTDIR}/exposure-ladder-{ZARR.stem}.mp4"
w = OverlayWriter(ladder_path, H, W, scale)


def emit(rate, native_start, exposure, label=None):
    """One output frame plus its fps / exposure / timestamp overlay."""
    w.write(rate, [
        (label or format_fps(NATIVE_FPS / exposure), 0.45),
        (f"exposure  {format_us(exposure * 1e6 / NATIVE_FPS, compact=True)}", 0.32),
        (f"t = {format_us(native_start * 1e6 / NATIVE_FPS)}", 0.32),
    ])


# --- 30 fps (real time): whole acquisition, 6/7 base bins per output frame ---
edges = np.round(np.linspace(0, nbins, 31)).astype(int)
for a, b in zip(edges[:-1], edges[1:]):
    n = (b - a) * BIN
    emit(base200[a:b].sum(0) / n, a * BIN, n, label="30 fps  (real time)")

# --- 200 fps: whole acquisition, one output frame per base bin ---
for b in range(nbins):
    emit(base200[b] / BIN, b * BIN, BIN)

# --- 1k / 10k / 50k / 100k fps: windows centred on the action ---
for exposure in (100, 10, 2, 1):
    start = CENTER - N_OUT * exposure // 2
    for k in range(N_OUT):
        s = start + k * exposure
        emit(accumulate(quanta, s, exposure) / exposure, s, exposure)

print(w.close(), "frames ->", ladder_path)

### 2. one exposure growing over time

A single exposure that keeps integrating over the frame slice, up to the full 96 ms.
Photon noise resolves into a clean image, then motion blur takes over. The exposures
matching familiar frame rates (10k / 1k / 120 / 60 / 30 fps) are added as exact stops
and held, so the video pauses on each. Two pacings, written to separate files so both
can be kept:

- `"constant"` — every output frame adds the same `STEP` native frames, so the exposure
  grows linearly. Noise falls as 1/sqrt(n), so nearly all the denoising happens in the
  first ~15 frames and the rest is motion blur building.
- `"geometric"` — the exposure grows by a fixed factor each step, from a single 10 us
  binary frame up. Spreads the denoising across the whole video, but the wall-clock rate
  of accumulation accelerates.

In [ ]:
# SLICE = (18_500, 28_100)   # the frame slice used elsewhere in this section
SLICE = (70000, 80000)   # the frame slice used elsewhere in this section
RATE = "geometric"          # "constant" or "geometric"
STEP = 40                  # constant: native frames added per output frame (400 us)
N_GEOM = 140               # geometric: steps from one binary frame to the whole slice
PAUSE_FPS = [10_000, 1_000, 120, 60, 30]   # hold when the exposure hits these rates
PAUSE = 20                 # output frames held at each of those
HOLD = 45                  # extra frames held on the final full exposure
DWELL = 1 if RATE == "constant" else 2     # output frames held per ordinary step

grow_path = f"{OUTDIR}/growing-exposure-{RATE}-{ZARR.stem}.mp4"
n_slice = SLICE[1] - SLICE[0]
if RATE == "constant":
    targets = np.arange(STEP, n_slice + 1, STEP)
else:
    targets = np.unique(np.round(np.logspace(0, np.log10(n_slice), N_GEOM)).astype(int))

# fold the familiar frame rates in as exact stops so the pauses land on round numbers
pause_at = {round(NATIVE_FPS / f) for f in PAUSE_FPS}
pause_at = {n for n in pause_at if 0 < n <= n_slice}
targets = np.union1d(targets, sorted(pause_at))
print(f"{len(targets)} {RATE} steps x{DWELL}, {len(pause_at)} pauses x{PAUSE}, "
      f"{HOLD} hold = {(len(targets) - len(pause_at)) * DWELL + len(pause_at) * (DWELL + PAUSE) + HOLD} frames")

data = np.asarray(quanta[SLICE[0]:SLICE[1]])   # ~2.5 GB, keeps the accumulation fast
w = OverlayWriter(grow_path, H, W, scale)

acc = np.zeros((H, W), dtype=np.float32)
count = 0
for n in targets:
    acc += data[count:n].sum(axis=0, dtype=np.float32)
    count = n
    w.write(acc / count, [
        (f"exposure  {format_us(count * 1e6 / NATIVE_FPS, compact=True)}", 0.45),
        (f"equivalent to  {format_fps(NATIVE_FPS / count)}", 0.32),
    ], repeat=DWELL + (PAUSE if n in pause_at else 0) + (HOLD if n == targets[-1] else 0))

print(w.close(), "frames ->", grow_path)